In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import json

# ============================================================
# CONFIGURATION
# ============================================================

RAW_DATA_DIR = Path("data/raw")
CLEANED_DATA_DIR = Path("data/cleaned")
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("🔍 Analyse des fichiers CSV bruts...")
print("=" * 60)

# ============================================================
# 1. CHARGEMENT DES DONNÉES
# ============================================================

csv_files = list(RAW_DATA_DIR.glob("wifi_*.csv"))

if not csv_files:
    print("❌ Aucun fichier CSV trouvé dans", RAW_DATA_DIR)
    print("Assurez-vous que vos fichiers sont au format: wifi_NomSalle.csv")
else:
    print(f"✅ {len(csv_files)} fichiers trouvés")

# Charger tous les CSV
dataframes = {}
all_bssids = []

for csv_file in csv_files:
    # Extraire le nom de la salle depuis le nom du fichier
    room_name = csv_file.stem.replace("wifi_", "")
    
    try:
        df = pd.read_csv(csv_file)
        
        # Vérifier les colonnes nécessaires
        required_cols = ['BSSID', 'SSID', 'RSSI(dBm)', 'Time']
        if not all(col in df.columns for col in required_cols):
            print(f"⚠️  Colonnes manquantes dans {csv_file.name}")
            continue
        
        # 🔧 NORMALISER LES BSSID (enlever les deux-points finaux si présents)
        df['BSSID'] = df['BSSID'].str.strip().str.rstrip(':')
        
        # 🔧 NORMALISER LES SSID (supprimer espaces)
        df['SSID'] = df['SSID'].str.strip()
        
        dataframes[room_name] = df
        all_bssids.extend(df['BSSID'].unique())
        
        print(f"📄 {room_name}: {len(df)} mesures, {df['BSSID'].nunique()} BSSID uniques")
        
    except Exception as e:
        print(f"❌ Erreur sur {csv_file.name}: {e}")

print("\n" + "=" * 60)

# ============================================================
# 2. IDENTIFICATION DES BSSID COMMUNS (STRATÉGIE ADAPTATIVE)
# ============================================================

print("\n🔍 Identification des BSSID les plus fréquents...")

# Compter dans combien de salles chaque BSSID apparaît
bssid_room_count = {}
for bssid in set(all_bssids):
    count = sum(1 for df in dataframes.values() if bssid in df['BSSID'].values)
    bssid_room_count[bssid] = count

# Trier par fréquence
bssid_sorted = sorted(bssid_room_count.items(), key=lambda x: x[1], reverse=True)

num_rooms = len(dataframes)

# Stratégie adaptative : garder les BSSID présents dans au moins X% des salles
min_room_percentage = 0.70  # 70% des salles minimum
min_rooms_required = max(int(num_rooms * min_room_percentage), 1)

common_bssids = [bssid for bssid, count in bssid_sorted if count >= min_rooms_required]

print(f"\n✅ {len(common_bssids)} BSSID retenus")
print(f"📊 Présents dans au moins {min_rooms_required}/{num_rooms} salles ({min_room_percentage*100:.0f}%)")

if len(common_bssids) == 0:
    print("\n⚠️  Aucun BSSID ne satisfait le critère de 70%")
    print("📉 Réduction du seuil à 50%...")
    
    min_room_percentage = 0.50
    min_rooms_required = max(int(num_rooms * min_room_percentage), 1)
    common_bssids = [bssid for bssid, count in bssid_sorted if count >= min_rooms_required]
    
    print(f"✅ {len(common_bssids)} BSSID retenus avec le seuil de 50%")

if len(common_bssids) == 0:
    print("\n❌ Toujours aucun BSSID commun !")
    print("📊 Distribution des BSSID par nombre de salles:")
    
    distribution = Counter(bssid_room_count.values())
    for num_rooms_present in sorted(distribution.keys(), reverse=True):
        print(f"   {num_rooms_present} salles: {distribution[num_rooms_present]} BSSID")
    
    print("\n💡 Suggestion: Prendre les BSSID présents dans au moins 1 salle")
    common_bssids = list(bssid_room_count.keys())
    print(f"✅ {len(common_bssids)} BSSID totaux seront utilisés")

# Afficher les BSSID les plus fréquents
print("\n📡 Top 20 BSSID les plus présents:")
for i, (bssid, count) in enumerate(bssid_sorted[:20], 1):
    status = "✅" if bssid in common_bssids else "❌"
    coverage = (count / num_rooms) * 100
    print(f"  {status} {i:2d}. {bssid} → {count}/{num_rooms} salles ({coverage:.0f}%)")

# ============================================================
# 3. NETTOYAGE ET FILTRAGE
# ============================================================

print("\n🧹 Nettoyage des données...")
print("=" * 60)

cleaned_dataframes = {}
cleaning_stats = {}

for room_name, df in dataframes.items():
    print(f"\n📍 Traitement de '{room_name}':")
    
    original_size = len(df)
    
    # 1. Filtrer uniquement les BSSID communs
    df_filtered = df[df['BSSID'].isin(common_bssids)].copy()
    after_filter = len(df_filtered)
    
    print(f"  ✂️  Filtrage BSSID: {original_size} → {after_filter} mesures")
    
    if after_filter == 0:
        print(f"  ⚠️  AUCUNE mesure conservée pour cette salle !")
        # Afficher les BSSID présents dans cette salle
        room_bssids = df['BSSID'].unique()[:5]
        print(f"  💡 BSSID présents: {', '.join(room_bssids)}")
        continue
    
    # 2. Supprimer les doublons exacts
    df_cleaned = df_filtered.drop_duplicates(
        subset=['BSSID', 'SSID', 'RSSI(dBm)', 'Time'],
        keep='first'
    )
    after_dedup = len(df_cleaned)
    
    duplicates_removed = after_filter - after_dedup
    print(f"  🗑️  Doublons supprimés: {duplicates_removed}")
    
    # 3. Ajouter la colonne Room
    df_cleaned['Room'] = room_name
    
    # 4. Réorganiser les colonnes
    df_cleaned = df_cleaned[['Room', 'BSSID', 'SSID', 'RSSI(dBm)', 'Time']]
    
    cleaned_dataframes[room_name] = df_cleaned
    
    cleaning_stats[room_name] = {
        'original': original_size,
        'after_filter': after_filter,
        'after_dedup': after_dedup,
        'duplicates_removed': duplicates_removed,
        'reduction_pct': round((1 - after_dedup/original_size) * 100, 1) if original_size > 0 else 0
    }
    
    print(f"  ✅ Résultat final: {after_dedup} mesures ({cleaning_stats[room_name]['reduction_pct']}% de réduction)")

# ============================================================
# 4. SAUVEGARDE DES FICHIERS NETTOYÉS
# ============================================================

print("\n💾 Sauvegarde des fichiers nettoyés...")
print("=" * 60)

for room_name, df_cleaned in cleaned_dataframes.items():
    output_file = CLEANED_DATA_DIR / f"cleaned_{room_name}.csv"
    df_cleaned.to_csv(output_file, index=False)
    print(f"✅ {output_file.name} ({len(df_cleaned)} lignes)")

# ============================================================
# 5. CRÉATION D'UN DATASET UNIFIÉ
# ============================================================

if cleaned_dataframes:
    print("\n📊 Création du dataset unifié...")

    df_unified = pd.concat(cleaned_dataframes.values(), ignore_index=True)
    unified_file = CLEANED_DATA_DIR / "dataset_unified.csv"
    df_unified.to_csv(unified_file, index=False)

    print(f"✅ {unified_file.name} créé ({len(df_unified)} lignes)")
else:
    print("\n❌ Aucune donnée nettoyée à sauvegarder !")

# ============================================================
# 6. RAPPORT FINAL
# ============================================================

print("\n" + "=" * 60)
print("📈 RAPPORT DE NETTOYAGE")
print("=" * 60)

total_original = sum(stats['original'] for stats in cleaning_stats.values())
total_final = sum(stats['after_dedup'] for stats in cleaning_stats.values())

print(f"\n📂 Salles traitées: {len(dataframes)}")
print(f"📡 BSSID retenus: {len(common_bssids)}")
print(f"📊 Seuil de couverture: {min_room_percentage*100:.0f}% des salles")
print(f"📊 Mesures totales:")
print(f"   - Original: {total_original}")
print(f"   - Après nettoyage: {total_final}")

if total_original > 0:
    reduction = round((1-total_final/total_original)*100, 1)
    print(f"   - Réduction: {reduction}%")

print("\n📋 Détails par salle:")
for room_name, stats in cleaning_stats.items():
    print(f"   {room_name}: {stats['original']} → {stats['after_dedup']} "
          f"(-{stats['reduction_pct']}%)")

# Analyse de couverture des BSSID
print("\n📡 Couverture des BSSID par salle:")
for room_name, df in cleaned_dataframes.items():
    coverage = (df['BSSID'].nunique() / len(common_bssids)) * 100
    print(f"   {room_name}: {df['BSSID'].nunique()}/{len(common_bssids)} BSSID ({coverage:.0f}%)")

# Sauvegarder le rapport
report = {
    'num_rooms': len(dataframes),
    'num_common_bssids': len(common_bssids),
    'common_bssids': common_bssids,
    'min_room_percentage': min_room_percentage,
    'total_original': total_original,
    'total_final': total_final,
    'room_stats': cleaning_stats,
    'bssid_coverage': {bssid: count for bssid, count in bssid_sorted[:50]}  # Top 50
}

report_file = CLEANED_DATA_DIR / "cleaning_report.json"
with open(report_file, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n💾 Rapport sauvegardé: {report_file.name}")
print("\n✅ Nettoyage terminé !")


🔍 Analyse des fichiers CSV bruts...
✅ 24 fichiers trouvés
📄 couloirB(2e): 5659 mesures, 57 BSSID uniques
📄 couloirB: 2308 mesures, 55 BSSID uniques
📄 couloirPbasA: 2198 mesures, 32 BSSID uniques
📄 couloirPbasB: 3016 mesures, 33 BSSID uniques
📄 P101A: 1720 mesures, 30 BSSID uniques
📄 P101B: 2480 mesures, 32 BSSID uniques
📄 P102A: 2841 mesures, 27 BSSID uniques
📄 P102B: 2437 mesures, 25 BSSID uniques
📄 P103A: 1058 mesures, 25 BSSID uniques
📄 PALIER1ER: 5715 mesures, 58 BSSID uniques
📄 PALIER2E: 5050 mesures, 53 BSSID uniques
📄 pallierPbas: 3628 mesures, 38 BSSID uniques
📄 S101A: 5645 mesures, 59 BSSID uniques
📄 S101B: 3479 mesures, 59 BSSID uniques
📄 S102A: 2746 mesures, 38 BSSID uniques
📄 S102B: 2811 mesures, 55 BSSID uniques
📄 S103B: 4902 mesures, 51 BSSID uniques
📄 S104: 5176 mesures, 53 BSSID uniques
📄 S202A: 1968 mesures, 25 BSSID uniques
📄 S202B: 2148 mesures, 26 BSSID uniques
📄 S203A: 4278 mesures, 49 BSSID uniques
📄 S203B: 3795 mesures, 47 BSSID uniques
📄 S204A: 1898 mesures, 29 

C:\Users\virgi\AppData\Local\Temp\ipykernel_65036\2366397173.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Room'] = room_name
C:\Users\virgi\AppData\Local\Temp\ipykernel_65036\2366397173.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Room'] = room_name
C:\Users\virgi\AppData\Local\Temp\ipykernel_65036\2366397173.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

Se